In [1]:
import os

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv
import os

load_dotenv()

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")
print("[OK] HuggingFace 키 설정됨" if os.getenv("HUGGINGFACE_API_KEY") else "[ERROR] HUGGINGFACE 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[OK] Google API 키 설정됨
[OK] LangSmith 키 설정됨
[OK] HuggingFace 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [3]:
import requests

url = "https://www.dropbox.com/scl/fi/ugue14fyo010jgc7wuh4g/teddynote-sample-video.mp4?rlkey=wcsiktklt7jgoibsluft3m6z9&st=prv4p2uu&dl=1"

response = requests.get(url)

with open("teddynote-sample-video.mp4", "wb") as f:
    f.write(response.content)

In [4]:
video_file_name = "teddynote-sample-video.mp4"

In [24]:
import os
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("파일 업로드 중...")

video_file = client.files.upload(
    file=video_file_name
)

print(f"업로드 완료: {video_file.uri}")

파일 업로드 중...
업로드 완료: https://generativelanguage.googleapis.com/v1beta/files/hz9ciweg8i4b


In [17]:
import time

while video_file.state.name == "PROCESSING":
    print("비디오 업로드 및 전처리가 완료될 때까지 잠시만 기다려주세요...")
    time.sleep(10)
    video_file = client.files.get(name=video_file.name)
    
if video_file.state.name == "FAILED":
    raise ValueError(f"비디오 처리 실패: {video_file.state.name}")

print("비디오 처리 완료, 대화 시작 가능")

비디오 처리 완료, 대화 시작 가능


In [21]:
prompt = "이 영상에 대해서 짧게 요약해 줄 수 있나요?"

response = client.models.generate_content(
    model= "gemini-3.5-flash",
    contents= [video_file, prompt]
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


이 영상은 **PDF 문서를 분석하여 텍스트, 표(Table), 이미지를 고품질로 추출하고 이를 마크다운(Markdown) 형태로 변환하는 고급 문서 전처리 기술**에 대한 세미나 영상입니다.

주요 내용을 요약하면 다음과 같습니다.

### 1. 기존 문서 전처리의 문제점
* 기존 파서나 OCR을 사용하면 PDF에서 단순 텍스트만 추출되어 **표가 깨지거나 이미지, 차트 정보가 누락**됩니다.
* 이로 인해 벡터 DB에 데이터를 저장하고 검색(Retrieval)할 때 검색 품질이 크게 떨어집니다.

### 2. 레이아웃 파싱 알고리즘 활용 (Upstage Layout Analyzer 등)
* PDF 문서의 레이아웃을 분석하여 제목, 본문, 이미지, 표 등의 요소를 분류하고 각각의 **바운딩 박스(Bounding Box) 좌표**를 획득합니다.
* 획득한 좌표 정보를 활용하여 원본 PDF에서 **해당 이미지와 표 영역만 고화질로 크롭(Crop)하여 개별 파일로 저장**합니다.

### 3. 고품질 마크다운(Markdown) 변환 및 시연
* 줄글 텍스트와 함께 크롭된 이미지, 표가 원래 위치에 맞게 마크다운 문법으로 결합되어 단일 파일로 변환되는 과정을 코드로 직접 시연합니다. 
* 다단 구성의 복잡한 PDF 문서도 가독성 높은 한 줄 구조의 마크다운으로 깔끔하게 변환됩니다.

### 4. LangGraph(랭그래프)를 통한 확장 및 벡터 DB 저장
* 대용량 PDF 처리를 위해 파일을 분할하고, 요소를 추출하여 병렬 배치(Batch) 처리로 요약본(Summary)을 신속하게 생성하는 워크플로우를 구성합니다.
* 이미지와 표에 대한 요약본, 태그 정보, 그리고 원본 이미지 파일 ID를 매핑하여 벡터 DB에 저장함으로써, **추후 RAG(검색 증강 생성) 시스템에서 멀티모달 검색 성능을 극대화**할 수 있습니다.


In [25]:
client.files.delete(name=video_file.name)

print(f"영상 삭제 완료: {video_file.uri}")

영상 삭제 완료: https://generativelanguage.googleapis.com/v1beta/files/hz9ciweg8i4b
